# Lipkin-Meshkov-Glick (LMG) Model

The Lipkin-Meshkov-Glick (LMG) model describes a collectively interacting
spin system in which each qubit can interact with the other qubits through
symmetric spin-spin interactions. Unlike models based only on nearest-neighbor
coupling, the LMG model uses collective interactions and provides a useful
framework for studying quantum correlations, entanglement, and many-body
effects.

The dynamics result from the competition between the collective spin-spin
interaction and an external transverse field. Even for a small number of
qubits, this competition can produce non-trivial quantum evolution and
coherence behavior.

The LMG Hamiltonian is given by:

$$
H =
-\frac{\lambda}{N}
\left(
S_x^2+\gamma S_y^2
\right)
-hS_z
$$

where $\lambda$ represents the interaction strength, $\gamma$ controls the
anisotropy between the $x$ and $y$ spin interactions, and $h$ represents the
strength of the external magnetic field.

The collective spin operators are defined by combining the contributions from
all individual qubits:

$$
S_\alpha =
\frac{1}{2}
\sum_{i=1}^{N}
\sigma_i^\alpha,
\qquad
\alpha \in \{x,y,z\}
$$

Using these collective operators, the model captures the combined behavior of
the spin system rather than treating each qubit as an independent particle.
The resulting interaction between collective spin dynamics and the external
field can generate quantum correlations and entanglement during the time
evolution.

### PAULI-BASIS MEASUREMENTS for LMG Model

In [ ]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_fez")

shots = # type in the number of shots depending up on your problem

total_time = # total evolution time
iterations = # number of density matrices (time points)
dt = total_time / iterations

# LMG parameters
lam =
gamma =
h =
N =

print("Backend:", backend.name)
print("dt per Trotter step:", dt)

# LMG 2-QUBIT MODEL CIRCUIT
def lmg_circuit(n, lam, gamma, h, dt):
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.h(1)

    for _ in range(n):

        # Sx^2 term
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(2 * lam * dt / N, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)

        # Sy^2 term
        qc.sdg(0)
        qc.sdg(1)
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(2 * lam * gamma * dt / N, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)
        qc.s(0)
        qc.s(1)

        # h Sz term
        qc.rz(2 * h * dt, 0)
        qc.rz(2 * h * dt, 1)

    return qc

# BASIS ROTATIONS + MEASUREMENT
def add_basis_and_measure(qc, basis):
    qc2 = qc.copy()
    for q, b in enumerate(basis):
        if b == "X":
            qc2.h(q)
        elif b == "Y":
            qc2.sdg(q)
            qc2.h(q)
    qc2.measure_all()
    return qc2

bases = ["ZZ","ZX","ZY","ZI",
         "XZ","XX","XY","XI",
         "YZ","YX","YY","YI",
         "IZ","IX","IY"]

all_circuits = []

for n in range(1, iterations + 1):
    for basis in bases:
        base = lmg_circuit(n, lam, gamma, h, dt)
        full = add_basis_and_measure(base, basis)
        all_circuits.append(full)

print("Total circuits submitted:", len(all_circuits))  # 75 Circuits

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(all_circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits, shots=shots)

print("\nSAVE THIS JOB ID:")
print(job.job_id())

## Retrieving the Job Results and Performing State Analysis

After submitting the quantum circuits to the IBM Quantum backend, the Job ID
printed by the execution code should be saved. The Job ID uniquely identifies
the submitted experiment and allows the experimental results to be retrieved
later without executing the quantum circuits again.

The saved Job ID is entered into the following analysis code:

```python
job = service.job("YOUR_JOB_ID")